In [1]:
import time
import csv
import re
from urllib.parse import urljoin
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.myfootdr.com.au"
REGIONS_URL = "https://www.myfootdr.com.au/our-clinics/regions/"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; myfootdr-scraper/1.0)"
}

# ------------- Helpers -------------

def get_soup(url):
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")

def clean_text(s):
    if not s:
        return ""
    return re.sub(r"\s+", " ", s).strip()

def extract_email(text, soup):
    # prefer mailto links
    for a in soup.select('a[href^="mailto:"]'):
        mail = a.get("href", "").replace("mailto:", "").strip()
        if "@" in mail:
            return mail
    m = re.search(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", text)
    return m.group(0) if m else ""

def extract_phone(text, soup):
    # tel: links
    for a in soup.select('a[href^="tel:"]'):
        tel = a.get("href", "").replace("tel:", "").strip()
        tel = tel.replace(" ", "")
        if tel:
            return tel
    # broad AU-style phone pattern
    m = re.search(r"\(?0?\d{1,4}\)?[\s-]?\d{3,4}[\s-]?\d{3,4}", text)
    return clean_text(m.group(0)) if m else ""

def extract_address(soup):
    # Common patterns: address tag, elements with 'address' in class, or marked itemprop
    candidates = []

    for tag in soup.select("address, [itemprop='address'], .address, .clinic-address"):
        txt = clean_text(tag.get_text(" ", strip=True))
        if txt:
            candidates.append(txt)

    # look for headings that say 'Address'
    for h in soup.find_all(["h2", "h3", "strong"]):
        if "address" in h.get_text(" ", strip=True).lower():
            sib = h.find_next_sibling()
            if sib:
                txt = clean_text(sib.get_text(" ", strip=True))
                if txt:
                    candidates.append(txt)

    # Heuristic if none found
    if not candidates:
        text = clean_text(soup.get_text(" ", strip=True))
        lines = [l.strip() for l in text.split(".") if l.strip()]
        aus_states = ["QLD", "NSW", "VIC", "SA", "WA", "TAS", "NT", "ACT"]
        for l in lines:
            if any(st in l for st in aus_states) or re.search(r"\b\d{4}\b", l):
                candidates.append(l)

    if not candidates:
        return ""

    # pick longest plausible
    candidates = [c for c in candidates if len(c) >= 10]
    candidates.sort(key=len, reverse=True)
    return candidates[0] if candidates else ""

def extract_services(soup):
    # look for headings mentioning services
    services = []
    heads = []
    for h in soup.find_all(["h2", "h3", "h4"]):
        text = h.get_text(" ", strip=True).lower()
        if any(k in text for k in ["services", "we treat", "we offer"]):
            heads.append(h)

    for head in heads:
        ul = head.find_next("ul")
        if ul:
            for li in ul.find_all("li"):
                t = clean_text(li.get_text(" ", strip=True))
                if t:
                    services.append(t)

    # fallback: pick likely podiatry-related bullet items anywhere
    if not services:
        keywords = [
            "orthotic", "ingrown", "nail", "diabetic", "plantar",
            "heel", "gait", "corn", "callus", "sports", "pediatric",
        ]
        for li in soup.select("ul li"):
            t = clean_text(li.get_text(" ", strip=True))
            if any(k in t.lower() for k in keywords):
                services.append(t)

    uniq = []
    seen = set()
    for s in services:
        if s not in seen:
            uniq.append(s)
            seen.add(s)
    return "; ".join(uniq)[:2000]

def extract_clinic_name(soup):
    h1 = soup.find("h1")
    if h1:
        return clean_text(h1.get_text(" ", strip=True))
    if soup.title and soup.title.string:
        return clean_text(soup.title.string)
    return ""

# ------------- Link discovery -------------

def get_region_links():
    soup = get_soup(REGIONS_URL)
    region_links = []
    for a in soup.select("a"):
        href = a.get("href") or ""
        if "/our-clinics/regions/" in href:
            url = urljoin(BASE_URL, href)
            region_links.append(url)
    # dedupe
    unique = []
    seen = set()
    for u in region_links:
        if u not in seen:
            unique.append(u)
            seen.add(u)
    return unique

def get_clinic_links_from_region(region_url):
    soup = get_soup(region_url)
    clinic_links = []
    for a in soup.select("a"):
        href = a.get("href") or ""
        # individual clinics are under /our-clinics/<slug>/
        if "/our-clinics/" in href and "/our-clinics/regions/" not in href:
            url = urljoin(BASE_URL, href.split("#")[0])
            clinic_links.append(url)
    # dedupe
    unique = []
    seen = set()
    for u in clinic_links:
        if u not in seen:
            unique.append(u)
            seen.add(u)
    return unique

# ------------- Clinic scraper -------------

def scrape_clinic(clinic_url):
    print(f"Scraping clinic: {clinic_url}")
    soup = get_soup(clinic_url)
    text = clean_text(soup.get_text(" ", strip=True))

    name = extract_clinic_name(soup)
    address = extract_address(soup)
    email = extract_email(text, soup)
    phone = extract_phone(text, soup)
    services = extract_services(soup)

    return {
        "Name of Clinic": name,
        "Address": address,
        "Email": email,
        "Phone": phone,
        "Services": services,
    }

# ------------- Main -------------

def main():
    region_links = get_region_links()
    print(f"Found {len(region_links)} regions.")

    all_clinic_urls = set()
    for rurl in region_links:
        try:
            print(f"Region: {rurl}")
            clinic_urls = get_clinic_links_from_region(rurl)
            for cu in clinic_urls:
                all_clinic_urls.add(cu)
            time.sleep(0.5)
        except Exception as e:
            print(f"[WARN] region failed: {rurl} -> {e}")

    print(f"Total unique clinic URLs: {len(all_clinic_urls)}")

    rows = []
    for i, curl in enumerate(sorted(all_clinic_urls)):
        try:
            row = scrape_clinic(curl)
            rows.append(row)
        except Exception as e:
            print(f"[WARN] clinic failed: {curl} -> {e}")
        time.sleep(0.5)

    out_file = "clinics.csv"
    cols = ["Name of Clinic", "Address", "Email", "Phone", "Services"]
    with open(out_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=cols)
        writer.writeheader()
        for r in rows:
            writer.writerow({c: r.get(c, "") for c in cols})

    print(f"Saved {len(rows)} clinics to {out_file}")

if __name__ == "__main__":
    main()


Found 12 regions.
Region: https://www.myfootdr.com.au/our-clinics/regions/brisbane/
Region: https://www.myfootdr.com.au/our-clinics/regions/gold-coast/
Region: https://www.myfootdr.com.au/our-clinics/regions/north-queensland/
Region: https://www.myfootdr.com.au/our-clinics/regions/central-queensland/
Region: https://www.myfootdr.com.au/our-clinics/regions/new-south-wales/
Region: https://www.myfootdr.com.au/our-clinics/regions/sunshine-coast/
Region: https://www.myfootdr.com.au/our-clinics/regions/victoria/
Region: https://www.myfootdr.com.au/our-clinics/regions/south-australia/
Region: https://www.myfootdr.com.au/our-clinics/regions/western-australia/
Region: https://www.myfootdr.com.au/our-clinics/regions/northern-territory/
Region: https://www.myfootdr.com.au/our-clinics/regions/tasmania/
Region: https://www.myfootdr.com.au/our-clinics/regions/
Total unique clinic URLs: 98
Scraping clinic: https://www.myfootdr.com.au/our-clinics/
Scraping clinic: https://www.myfootdr.com.au/our-clin